# BHARAT Red Team: QLoRA Fine-Tuned dolphin-llama3:8b vs Qwen3:8b

**Fine-Tuning:** QLoRA (4-bit NF4 quantization + LoRA rank=32, alpha=64)

**Blue Model:** `qwen3:8b` (defensive) | **Red Model:** `dolphin-llama3:8b` (fine-tuned attacker)

**Attacks:** 20 Single-Turn + Multi-Turn Memory Poisoning

**Inspired by:** Obliteratus-style prompts + BHARAT attack strategies

| Component | Detail |
|---|---|
| Base Model | cognitivecomputations/dolphin-2.9-llama3-8b |
| Quantization | 4-bit NF4 (BitsAndBytes) |
| LoRA Rank | 32 |
| LoRA Alpha | 64 |
| Training Epochs | 5 |
| Training Samples | 25+ adversarial prompt pairs |

## Step 1: Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate peft bitsandbytes trl huggingface_hub
!pip install -q openpyxl requests llama-cpp-python
print('All dependencies installed')

## Step 2: Start Ollama & Pull Base Models

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
proc = subprocess.Popen(['ollama','serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started')

In [ ]:
!ollama pull qwen3:8b
print('qwen3:8b ready (Blue Agent)')

In [ ]:
!ollama pull dolphin-llama3:8b
print('dolphin-llama3:8b ready (Red base model)')

## Step 3: Load Full Pipeline Code

In [ ]:
# Upload bharat_finetuned_attack.py to Colab first, or run this cell
# which contains the complete pipeline inline
import sys, os
# Option A: If you uploaded the .py file
if os.path.exists('bharat_finetuned_attack.py'):
    exec(open('bharat_finetuned_attack.py').read())
    print('Pipeline loaded from .py file')
else:
    print('Please upload bharat_finetuned_attack.py first')
    print('Or paste the code in the next cell')

## Step 4: Run QLoRA Fine-Tuning

This trains the red model on adversarial prompt-response pairs.
Takes ~15-20 minutes on a T4 GPU.

In [ ]:
# Fine-tune dolphin-llama3:8b with QLoRA
merged_path = finetune_qlora()
print(f'Merged model at: {merged_path}')

## Step 5: Convert to GGUF & Register in Ollama

In [ ]:
# Convert fine-tuned model to GGUF for Ollama
red_model = convert_to_ollama(merged_path)
print(f'Fine-tuned model: {red_model}')

## Step 6: Run All Attacks vs Qwen3:8b

Phase 1: 20 single-turn strategies (enhanced by fine-tuned red LLM)

Phase 2: Multi-turn memory poisoning (5 turns x 5 targets)

In [ ]:
# Run the full attack pipeline
results, elapsed = run_attacks(red_model)
print(f'\nCompleted {len(results)} attacks in {elapsed:.0f}s')

## Step 7: Generate ASR Report & Save All Files

In [ ]:
# Generate reports: Excel, CSV, JSONL, JSON metrics, Summary
metrics = generate_reports(results, elapsed, red_model)
print(f'\nOverall ASR: {metrics["overall_asr"]}%')

## Step 8: Download Results

In [ ]:
from google.colab import files
for f in ['Bharat_ASR_FineTuned.xlsx', 'bharat_verdicts_finetuned.jsonl',
          'bharat_metrics_finetuned.json', 'bharat_results_finetuned.csv',
          'Bharat_ASR_FineTuned_Summary.txt']:
    try:
        files.download(f)
        print(f'Downloaded: {f}')
    except Exception as e:
        print(f'Saved locally: {f}')